## Preparação do Ambiente

### Instalação de Pacotes

In [ ]:
! pip install --upgrade -q pandas numpy xgboost scikit-learn imbalanced-learn matplotlib optuna plotly nbformat kaleido

### Importação de Bibliotecas

In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import xgboost as xgb
from xgboost import XGBClassifier

import optuna
import pickle
import optuna.visualization as vis

from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

### Constantes

#### Local

In [ ]:
import sys
from pathlib import Path

# adiciona a pasta src ao sys.path
SRC_DIR = Path().resolve().parent  # .../src
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from utils.constants import *

#### Kaggle

In [ ]:
CATEGORICAL_COLUMNS = {
    "gestante_paciente", "raca_cor_paciente", "sigla_uf_residencia"
}

RANDOM_STATE = 42
TEST_RATIO = 0.15
N_FOLDS = 5
N_CLASSES = 3

TARGET_NAMES = ["low_risk", "alarm", "severe"]
TARGET_NAMES_COARSE = ["low_risk", "high_risk"]
TARGET_NAMES_FINE = ["alarm", "severe"] 


TARGET_LABEL_MAP = {name: idx for idx, name in enumerate(TARGET_NAMES)}
LABEL_TARGET_MAP = {idx: name for idx, name in enumerate(TARGET_NAMES)}

DATASET_GB_PATH = "../input/sinan-sbcas/dataset-processed-gb.csv"

## Carregamento dos Dados

In [ ]:
df = pd.read_csv(DATASET_GB_PATH)

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

## Treinamento com Validação Cruzada

In [ ]:
def train_on_folds(params):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full.iloc[train_idx], y_train_full.iloc[train_idx]
        X_valid, y_valid = X_train_full.iloc[valid_idx], y_train_full.iloc[valid_idx]

        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        # XGBoost with device='cuda' will handle data transfer
        # No need for explicit cp.asarray here, as XGBoost can process pandas DFs directly
        # when enable_categorical=True and device='cuda'.
        # The data will implicitly be moved to GPU by XGBoost itself.

        model = XGBClassifier(**params)

        try:
            model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            sample_weight=sample_weights,
            verbose=False
            )
            preds = model.predict(X_valid)
            
            # If GPU was used for training, predictions are still on GPU (if applicable).
            # Need to convert y_valid to numpy for f1_score.
            # XGBoost's predict for DMatrix outputs numpy arrays by default if not on GPU, but with GPU it might be CuPy.
            # Ensure y_valid is numpy for comparison.
            y_valid_np = y_valid.to_numpy() if isinstance(y_valid, pd.Series) else y_valid

            f1 = f1_score(y_valid_np, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0

    return np.mean(scores), np.std(scores)


## Otimização de Hiperparâmetros

In [ ]:
FIXED_PARAMS = {
    "n_estimators": 2000,
    "early_stopping_rounds": 5,
    "enable_categorical": True,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "device": "cuda",
    "eval_metric": "mlogloss",
    "random_state": RANDOM_STATE,
    "verbosity": 0
}

### Definição da Função Objetivo

In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 10, 16),
        
        # Afunilando o Learning Rate ao redor do 0.09
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.15),
        
        # Testando amostragem alta, mas permitindo leve ruído
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        
        # Reduzindo o range de regularização agressiva
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 15),
        'reg_lambda': trial.suggest_float('reg_lambda', 1, 30),
        
        # Estabilizando o peso mínimo do filho
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 12),
        
        # Mantendo Gamma baixo para não travar o crescimento
        'gamma': trial.suggest_float('gamma', 0, 2.0),
        
        **FIXED_PARAMS
    }

    avg, _ = train_on_folds(params)
    
    return avg

### Otimização com Optuna

In [ ]:
study = optuna.create_study(study_name="xgboost_study_cuda", direction="maximize")
study.enqueue_trial({'max_depth': 14, 'learning_rate': 0.091, 'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_alpha': 10, 'reg_lambda': 2, 'min_child_weight': 10, 'gamma': 0.0})
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

In [ ]:
output_dir = Path('results/optimization/xgboost/multi')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

### Visualizações

In [ ]:
display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [ ]:
vis.plot_param_importances(study).write_image(output_dir / 'opt_param_importances.png')
vis.plot_optimization_history(study).write_image(output_dir / 'opt_hist.png')

### Melhor Modelo

In [ ]:
BEST_PARAMS = {
    'max_depth': 16, 
    'learning_rate': 0.107893597297338, 
    'subsample': 0.9632988872870645, 
    'colsample_bytree': 0.9646675197441156, 
    'reg_alpha': 0.5877393757763218,
    'reg_lambda': 22.37101441077761, 
    'min_child_weight': 6, 
    'gamma': 0.11410499025763864
}

In [ ]:
final_params = {**BEST_PARAMS, **FIXED_PARAMS}

### Média e Desvio Padrão na Validação Cruzada

In [ ]:
avg_f1_final, std_f1_final = train_on_folds(final_params)

In [ ]:
print(avg_f1_final)
print(std_f1_final)

## Avaliação Final

### Treinamento e Predição

In [ ]:
if 'early_stopping_rounds' in FIXED_PARAMS:
    del FIXED_PARAMS['early_stopping_rounds']

if 'n_iter_no_change' in FIXED_PARAMS:
    del FIXED_PARAMS['n_iter_no_change']

In [ ]:
# Recalcular pesos para o conjunto de treino completo
# (Mantendo a lógica usada dentro do train_on_folds)
sample_weights_full = compute_sample_weight(class_weight='balanced', y=y_train_full)

final_model = XGBClassifier(**final_params)

final_model.fit(
    X_train_full, 
    y_train_full,
    sample_weight=sample_weights_full,
    verbose=False
)

y_pred_test = final_model.predict(X_test)

### Resultados

In [ ]:
print("Relatório de Classificação (Conjunto de Teste):")
print(classification_report(y_test, y_pred_test, target_names=TARGET_NAMES))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, 
    y_pred_test, 
    display_labels=TARGET_NAMES,
    cmap='Blues',
    normalize='true',
    ax=ax
)
ax.set_title("Matriz de Confusão - XGBoost (Multiclasse)")
plt.show()